### building simple LLM based application using LangGraph

In [3]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [4]:
load_dotenv()  # Load environment variables from .env file

True

In [6]:
LLM = ChatGoogleGenerativeAI(model="gemini-3.8-flash", api_key=os.getenv("GOOGLE_API_KEY"))

In [7]:
# create a state graph
class LLMState(TypedDict):
    input_text: str
    output_text: str

In [8]:
# define LLM_question node
def LLM_question(state: LLMState) -> LLMState:
    # extracting the question
    question = state['input_text']

    # form a prompt for the LLM
    prompt = f"Answer the following question: {question}"

    # get the answer from the LLM
    answer = LLM.invoke(prompt).content

    # update the state with the answer
    state['output_text'] = answer
    return state

In [10]:
# create a graph
graph = StateGraph(LLMState)

# add node
graph.add_node('llm_qa', LLM_question)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile the graph
workflow = graph.compile()

In [12]:
# Execute the state graph
intial_state  = {'input_text': "What is the capital of France?"}
final_state = workflow.invoke(intial_state)
print(final_state['output_text'])  # Output the answer from the LLM

[{'type': 'text', 'text': 'The capital of France is **Paris**.', 'extras': {'signature': 'Es4CCssCAWkUfRPtB71F7W+FcfI4g1BjSGPR29FmiiQz+6pCEfeNjFwZR2Z9C1FHAxYe50TMZR5QxU5180jnp1R+Sd92FKXeVUEFObLo5eYgDHXQMiMKlF1z/jX/fPhddoYVs9KxyTfINFNEXsKgXiO28/7fVYOSz9LQypVzMb5Ei7pdaurMCbaiRt4vmYV/G2F7Krn76whdeJfVvNlX9KFoF+pRZKikdD1jLis+uEhLaqgePiU4hvh/tVSBxxC53MpJ2FrIPDEbeiZFFu94HcPkseALLKKh1CY/g/3Cti8fa8ClgMttOt6hvW+jD6wdrUHd93usJMFlE/b6++qavw5eUXV4Nf81iyXop6PEJS+kMlFvi0cBkv8MuKWCDlibGWujRBe+hvDXkz4DzcsFrvXXjimgJAN6j7zOtqKVDIdzTnG8L7y/20lMULkb7Xhzaw=='}}]


In [13]:
# print capital of france withouting using LangGraph concept
answer = LLM.invoke("What is the capital of France?").content
print(answer)  # Output the answer from the LLM

[{'type': 'text', 'text': 'The capital of France is **Paris**.', 'extras': {'signature': 'EusCCugCAWkUfRN/YYOf4ksg8sqeLiw72GgsLvkGoTLwH2hPgIcKjPH9iuttZJKxGJCzcn/1j5RprPua426zzYLe+tDylLKbNQOYLEnJN2XtEVdwB7/4SLz3sWseDGyyFrlS//oloE4sve/E1tSoViKhF5snqQPpgy92Ji3ysVFWz8LyV4MjVGTPftyu60vP9vQo6zI+z+Hrfod2xcVjm/F6dqWeEQpVRpczLupKQ764Kfx1qwXH2R3Z32/dOAUP9UNyum4ouXX2rVpWe3YdezjdgWz/OXuX19Zm/KxGDpazFNVw7BwHlk8NI8XUPqA4CUvP5k9TTPo0L53/fHN/DH96DZ1AlwmUMeSvfEFmqiNQHvBpSBTrUyVHfgKfHF9s38/hgg3yNk5cmPQ6MP+7cTg5m4zdWOjyy8HgHpJJqj50QZwPAAb3sxMYrdtdOfnzqqiICLnsZCLo2g97H3BupUT1XJPB5JTM83Acd67s'}}]
